# Softmax Regression

Predicts the probability for a given k-class.

**Inputs**
- $X \in [B, D_{in}]$
- $y \in [B, k]$

**Model**

$z = X W + b \in [B, k]$

$f(z_j) = softmax(z_j) = \frac{e^{z_j}}{\sum_1^k e^{z_h}}$ predicts the probability of a given class $j \in [1, k]$

**Loss**

Cross entropy

$\mathbb{l} = -\frac{1}{B} \sum_{i, j} y_{i, j} \log f(z_{i, j})$

or in matrix format:

$\mathbb{l} = -\frac{1}{B} y^T \log f(z)$

Cross entropy for one class

$\mathbb{l_i} = -\frac{1}{B} \sum_{j} y_{j} \log f(z_{j})$


## Gradient Derivation ##

$$
\frac{d\mathbb{l}}{dW} = \frac{\partial l}{\partial f} \frac{\partial f}{\partial z} \frac{\partial z}{\partial W}
$$

$$
\frac{d\mathbb{l}}{db} = \frac{\partial l}{\partial f} \frac{\partial f}{\partial z} \frac{\partial z}{\partial b}
$$

Use the following derivatives:

### Derivative of loss function ###
$$\frac{\partial \mathbb{l}}{\partial f} = -\frac{1}{B}\frac{y}{f}$$




### Derivatives of softmax function ###
$\frac{\partial f}{\partial z}$ is a Jacobian matrix of [k, k], as

$$
\begin{Bmatrix}
\frac{\partial f_{0}}{\partial z_{0}}, &\frac{\partial f_{0}}{\partial z_{1}}, &..., &\frac{\partial f_{0}}{\partial z_{k-1}} \\
\frac{\partial f_{1}}{\partial z_{0}}, &\frac{\partial f_{1}}{\partial z_{1}}, &..., &\frac{\partial f_{1}}{\partial z_{k-1}} \\
..., &..., &..., &... \\
\frac{\partial f_{k-1}}{\partial z_{0}}, &\frac{\partial f_{k-1}}{\partial z_{1}}, &..., &\frac{\partial f_{k-1}}{\partial z_{k-1}} \\
\end{Bmatrix}
$$

For given element $\frac{\partial f_i}{\partial z_j}$:

- Case 1: $i == j$

    $$
    \begin{aligned}
    \frac{\partial f_i}{\partial z_i} &= \frac{\partial \frac{e^{z_i}}{\sum_1^k e^z_h}}{\partial z_i} \\
        &=\frac{e^{z_i}}{\sum_1^k e^{z_h}} - (\frac{e^{z_i}}{\sum_1^k e^{z_h}})^2 \\
        &= f_i - f_i^2 \\
        &= f_i(1-f_i) \\
    \end{aligned}
    $$

- Case 2: $i \neq j$

    $$
    \begin{aligned}
    \frac{\partial f_i}{\partial z_j} &= \frac{\partial \frac{e^{z_i}}{\sum_1^k e^z_h}}{\partial z_j} \\
        &=0 - \frac{e^{z_i}e^{z_j}}{(\sum_1^k e^{z_h})^2} \\
        &= -f_if_j\\
    \end{aligned}
    $$

Combing these two cases together:

$$
\frac{\partial f_i}{\partial z_j} = f_i(\delta_{i,j} - f_j)
$$
where $\delta_{ij} = 1$ if $i == j$ else 0.





### Derivatives of Linear Function ###
$$
\frac{\partial z}{\partial W} = X
$$

$$
\frac{\partial z}{\partial b} = 1|_{(B \times 1)}
$$


### Final Gradients ###
One can simply multiplying the above parts together following the gradient chain rule to get the final gradients. 

But for this specific application, calculating the Jacobian may not be efficient if the matrix is big. This calculation can be further improved by removing the needs of Jacobian calculation. 

For a single class j:

$$
\begin{aligned}
\frac{\partial \mathbb{l}}{\partial z_j} = \frac{\partial \mathbb{l}}{\partial f} \frac{\partial f}{\partial z_j} \\
    &=-\frac{1}{B} \frac{y}{f} f(\delta_{i, j} - f)

\end{aligned}
$$


## Data and Settings

In [14]:
import torch 
import numpy as np

In [15]:
B, Di, Do = 100, 10, 1

X = np.random.randn(B, Di)
y = np.random.randn(B, Do)
y = (y >= 0.5).astype(np.float32)

max_iters = 1000
lr = 1e-2

## Numpy Implementation

In [16]:
def model(x, W, b):
    """
    Args:
        x: (B, Di)
        W: (Di, Do)
        b: (1,)
    Returns:
        (B, Do)
    """
    z = x @ W + b
    f = 1 / (1 + np.exp(-z))
    
    return f

def binary_cross_entropy(pred, target):
    """
    Args:
        pred: (B, Do)
        target: (B, Do)
    Returns:
        scalar
    """
    eps = 1e-8
    loss = - (target * np.log(pred + eps) + (1 - target) * np.log(1 - pred + eps))
    return np.mean(loss)

def gradients(pred, target, x):
    """
    Args:
        pred: (B, Do)
        target: (B, Do)
        x: (B, Di) 
    """
    B, _= pred.shape
    dw = 1/B * x.T @ (pred - target)
    db = 1/B * np.sum(pred - target)
    return dw, db


In [17]:
# TRAINING 
# initialize weights
W = np.random.randn(Di, Do)
b = np.random.randn(1)

for i in range(max_iters):
    # forward 
    pred = model(X, W, b)

    # loss 
    loss = binary_cross_entropy(pred, y)

    # backward for gradient computation
    dw, db = gradients(pred, y, X)

    # update weights
    W -= lr * dw
    b -= lr * db

    # log 
    if i % 100 == 0:
        print(f"iter {i}: loss {loss:.4f}")

iter 0: loss 1.7398
iter 100: loss 1.5057
iter 200: loss 1.3022
iter 300: loss 1.1287
iter 400: loss 0.9844
iter 500: loss 0.8680
iter 600: loss 0.7775
iter 700: loss 0.7098
iter 800: loss 0.6612
iter 900: loss 0.6275
